In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn import metrics
from sklearn import model_selection
from sklearn import preprocessing

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.base import BaseEstimator, TransformerMixin

from sklearn import set_config
set_config(transform_output="pandas")

In [2]:
def isNan(v):
    return v is None or str(v) == 'nan' or str(v).strip() == ''

In [3]:
def valores_mas_comunes(array, col, n):
    columna = array[:, col]
    
    conteo = {}
    
    for x in columna:
        if x is None or str(x) == 'nan' or str(x).strip() == '':
            continue
        
        if x in conteo:
            conteo[x] += 1
        else:
            conteo[x] = 1
    
    resultado = sorted(conteo.items(), key=lambda x: x[1], reverse=True)
    
    if n:
        return resultado[:n]
    
    return resultado

In [4]:
def contar_vacios(array, col):
    columna = array[:, col]
    
    contador = 0
    
    for x in columna:
        if x is None or str(x) == 'nan' or str(x).strip() == '':
            contador += 1
    
    return contador

## PREPROCESAMIENTO

In [5]:
class limpiar_espacios(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X).copy()

        for col in X_df.columns:
            # Comprobamos si la columna es un string ('object')
            if X_df[col].dtype == 'object':
                es_string = X_df[col].apply(lambda x: isinstance(x, str))
                
                X_df.loc[es_string,col] = X_df.loc[es_string,col].astype(str).str.strip().str.replace(r'\s+', ' ', regex=True)
                
        return X_df

In [6]:
class minusculizar(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X).copy()

        for col in X_df.columns:
            # Comprobamos si la columna es un string ('object')
            if X_df[col].dtype == 'object':
                es_string = X_df[col].apply(lambda x: isinstance(x, str))
                
                X_df.loc[es_string,col] = X_df.loc[es_string,col].str.lower()
                
        return X_df

In [7]:
class quitar_simbolos(BaseEstimator, TransformerMixin):    
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X).copy()
        

        X_df = X_df.replace(to_replace=r'[^a-zA-Z0-9áéíóúÁÉÍÓÚñÑ\s]', value='', regex=True)
               
        return X_df

In [8]:
class quitar_tildes(BaseEstimator, TransformerMixin): 
    def __init__(self):
        self.reemplazos = {
            'á':'a','à':'a','ä':'a','â':'a','ã':'a','å':'a',
            'é':'e','è':'e','ë':'e','ê':'e',
            'í':'i','ì':'i','ï':'i','î':'i',
            'ó':'o','ò':'o','ö':'o','ô':'o','õ':'o',
            'ú':'u','ù':'u','ü':'u','û':'u',
            'ñ':'n',
            'ç':'c',
            'ß':'ss'
        }
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X).copy()

        
        X_df = X_df.replace(self.reemplazos, regex=True)
               
        return X_df

In [9]:
class agrupar_por_inclusion(BaseEstimator, TransformerMixin): 
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X).copy()

        for col in X_df.columns:

            if X_df[col].dtype != 'object':
                continue
            # Mantenemos referencia entre el valor original y el modificado
            v_limpios = {}

            for v in X_df[col]:
                # Eliminamos espacios innecesarios
                v_str = str(v).strip()

                # Sino es nan lo guardamos
                if v_str and v_str.lower() != 'nan':
                    v_limpios[v_str] = v

            # Ordenamos las keys(valores originales) por longitud 
            v_ordenados = sorted(v_limpios.keys(), key=len)
    
            mapeo = {}
            procesados = set()
            originales = []

            for i,corto in enumerate(v_ordenados):
                # Si ya hemos procesado la key no la volvemos a procesar
                if corto in procesados:
                    continue

                # Para solo acortar strings de 2 palabras
                if len(corto.split()) < 2:
                    continue

                # Miramos los siguientes
                for j in range(i + 1, len(v_ordenados)):
                    largo = v_ordenados[j]

                    # Si ese valor largo no ha sido procesado 
                    if largo not in procesados and corto in largo:
                        mapeo[largo] = corto
                        procesados.add(largo)
                        originales.append(largo)

            # Aplicamos mapeo de forma vectorizada
            def aplicar_mapeo(val):
                val_str = str(val).strip()
                
                # Imputamos el valor que le toca si existe
                return mapeo.get(val_str, val) 

            X_df[col] = X_df[col].apply(aplicar_mapeo)
            
        return X_df

In [10]:
def numerar_columna(array,columna):
    for i in range(array.shape[columna]):
        array[i][columna] = i + 1

In [11]:
def guardar_y(array):
    y = []
    for i in range(array.shape[0]):
        y.append(array[i][17])  
    return y

In [12]:
def copiar_columna(origen, destino, columna1, columna2):
    for i in range(origen.shape[0]):
        destino[i][columna2] = origen[i][columna1]

In [13]:
class rellenar_secuencial(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X).copy()

        for col in X_df.columns:
            if X_df[col].dtype == 'object':
                
                X_df[col] = X_df[col].ffill()
                    
        return X_df

In [14]:
class rellenar_con_moda(BaseEstimator, TransformerMixin):
    def __init__(self):
        # Definimos aquí qué consideramos "falsos nulos"
        self.valores_nulos = ["nan", "NaN", "null", "None", "", " ", None]
    
    def fit(self, X, y=None):
        X_df = pd.DataFrame(X).copy()

        # Reemplaza los falsos nan por nans de verdad para que pandas los pueda tratar de verdad
        X_df.replace(self.valores_nulos, np.nan, inplace=True)
        
        self.modes_ = X_df.mode(axis=0).iloc[0]
        
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X).copy()

        # Rellenamos falsos nans con nans de verdad para que verdad para que Pandas los pueda tratar
        # y usar sus funciones todo guapas
        X_df.replace(self.valores_nulos, np.nan, inplace=True)

        for col in X_df.columns:            
            # Rellenamos los nans que ahora son de verdad con la moda
            X_df[col] = X_df[col].fillna(self.modes_[col])
             
        return X_df
        

In [15]:
class rellenar_con_constante(BaseEstimator, TransformerMixin):
    def __init__(self):
        # Definimos aquí qué consideramos "falsos nulos"
        self.valores_nulos = ["nan", "NaN", "null", "None", "", " ", None]
    
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X).copy()

        # Rellenamos falsos nans con nans de verdad para que verdad para que Pandas los pueda tratar
        # y usar sus funciones todo guapas
        X_df.replace(self.valores_nulos, np.nan, inplace=True)

        for col in X_df.columns:
            
            # Si es categorico
            if X_df[col].dtype == 'object':
                X_df[col] = X_df[col].fillna("unknown")
                
             # Si es numerico
            else:
                X_df[col] = X_df[col].fillna(0)
                    
        return X_df

In [16]:
class rellenar_moda_por_grupos(BaseEstimator, TransformerMixin):
    def __init__(self, col_agrupacion, cols_a_imputar):
        # Definimos aquí qué consideramos "falsos nulos"
        self.valores_nulos = ["nan", "NaN", "null", "None", "", " ", None]

        self.col_agrupacion = col_agrupacion
        self.cols_a_imputar = cols_a_imputar
    
    def fit(self, X, y=None):
        X_df = pd.DataFrame(X).copy()

        # Reemplaza los falsos nan por nans de verdad para que pandas los pueda tratar de verdad
        X_df.replace(self.valores_nulos, np.nan, inplace=True)

        #Guardamos moda general por si el valor de agrupar que nos venga no existe
        self.modes_gen_ = X_df.mode(axis=0).iloc[0]

        self.modes_ = {}

        for col_agrup, col_input in zip(self.col_agrupacion,self.cols_a_imputar):
            
            def moda_segura(x):
                m = x.mode()
                return m.iloc[0] if not m.empty else np.nan
                
            self.modes_[col_input] = X_df.groupby(col_agrup)[col_input].apply(moda_segura)
        
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X).copy()

        # Rellenamos falsos nans con nans de verdad para que verdad para que Pandas los pueda tratar
        # y usar sus funciones todo guapas
        X_df.replace(self.valores_nulos, np.nan, inplace=True)

        for col_agrup, col_input in zip(self.col_agrupacion,self.cols_a_imputar):           
            modas = self.modes_[col_input]

            # Crea una Serie donde en cada indice pone la moda de su grupo
            modas_mapeadas = X_df[col_agrup].map(modas)

            # Si un elemento fuera nan, imputa su elemento correspondiente
            # en modas_mapeadas (que sera justo la moda de su grupo)
            X_df[col_input] = X_df[col_input].fillna(modas_mapeadas)

            #Rellenamos los nans que hubiesen quedado
            X_df[col_input] = X_df[col_input].fillna(self.modes_gen_[col_input]) 
                         
        return X_df
        

## EJEMPLO

In [18]:
data = pd.read_csv("data/fake_job_postings.csv")
print("CSV leido")

semilla = 1111

X = data.iloc[:,:-1]
y = data.iloc[:,-1]

X_resto, X_test, y_resto, y_test = model_selection.train_test_split(X, y, random_state = semilla, stratify = y, test_size = 0.1)
X_train, X_val, y_train, y_val = model_selection.train_test_split(X_resto, y_resto, random_state = semilla, stratify = y_resto, test_size = 0.2)

mg_agrup3 = ["employment_type","industry"]
mg_input3 = ["required_experience","function"]

mg_agrup4 = ["required_experience"]
mg_input4 = ["required_education"]

transformer1 = ColumnTransformer(transformers = [
    ("agruparInclusion",agrupar_por_inclusion(),["employment_type","required_experience","required_education","industry"]),
    ("rellenarSecuencial",rellenar_secuencial(),["salary_range"])],
                                remainder='passthrough',# Para que no destruya las colmnas que no toque
                                verbose_feature_names_out=False # Para que lo que devuelva sea legible (pone unas movidas que flipas)
                               )


transformer2 = ColumnTransformer(transformers = [
    ("rellenarModa",rellenar_con_moda(),["title","department","employment_type"]),
    ("rellenarConstante",rellenar_con_constante(),["industry"])],
                                remainder='passthrough',# Para que no destruya las colmnas que no toque
                                verbose_feature_names_out=False # Para que lo que devuelva sea legible (pone unas movidas que flipas)
                               )

transformer3 = ColumnTransformer(transformers = [
    ("rellenarModaGrupos",rellenar_moda_por_grupos(
        col_agrupacion = mg_agrup3, 
        cols_a_imputar = mg_input3),
     ["required_experience","employment_type","function","industry"])
    ],
                                remainder='passthrough',# Para que no destruya las colmnas que no toque
                                verbose_feature_names_out=False # Para que lo que devuelva sea legible (pone unas movidas que flipas)
                               )

transformer4 = ColumnTransformer(transformers = [
    ("rellenarModaGrupos",rellenar_moda_por_grupos(
        col_agrupacion = mg_agrup4, 
        cols_a_imputar = mg_input4),
     ["required_experience","required_education"])],
                                remainder='passthrough',# Para que no destruya las colmnas que no toque
                                verbose_feature_names_out=False # Para que lo que devuelva sea legible (pone unas movidas que flipas)
                               )
    
pipe = Pipeline([
    ("limpiaEspacios",limpiar_espacios()),
    ("minusculizar",minusculizar()),
    ("quitarSimbolos",quitar_simbolos()),
    ("quitarTildes",quitar_tildes()),
    ("transformer1", transformer1),
    ("transformer2", transformer2),
    ("transformer3", transformer3)
])



pipe.fit(X_train,y_train)
pipe.transform(X_val)


CSV leido


,required_experience,employment_type,function,industry,title,department,company_profile,required_education,salary_range,job_id,location,description,requirements,benefits,telecommuting,has_company_logo,has_questions
4998,midsenior level,fulltime,product management,unknown,product manager,sales,recombine provides clinical genetic testing ca...,NaN,NaN,4999,us ny new york,recombine seeks to develop the most comprehens...,about you masters or phd in genetics genetic c...,participate and contribute to an environment w...,0,1,0
1523,midsenior level,fulltime,marketing,computer software,head of marketing,marketing,tradables award winning platform helps brokers...,NaN,NaN,1524,dk 84 copenhagen,we are looking for a marketing leader to join ...,requirementsexcellent englishprevious marketin...,what we offerhandson experiencecompetitive sal...,0,1,1
5429,not applicable,fulltime,administrative,unknown,sheffield estate agents office assistant appre...,sales,established on the principles that full time e...,high school or equivalent,NaN,5430,gb sheffield,under the national apprenticeship scheme you m...,1618 year olds only due to government fundingf...,career prospects,0,1,1
11478,associate,fulltime,training,information technology and services,training manager,sales,sli systems is a saas company revolutionizing ...,bachelors degree,7000080000,11479,us ca san jose,our director of training needs a rockstar trai...,educationbabs required no exceptionsfocus in ...,were not your ordinary company we provides si...,0,1,0
13225,midsenior level,fulltime,marketing,unknown,local support compliance and fraud detection,sales,url31fdc354999cbb96507ebbe4e9c4aa7eed5edd0dd1c...,NaN,7000080000,13226,fi es helsinki,your initial work will be providing support to...,fluent finnish native preferred for communicat...,NaN,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6431,midsenior level,fulltime,marketing,unknown,team lead,sales,i28 technologies has demonstrated expertise in...,NaN,2500035000,6432,us nj trenton,job descriptionposition title team lead trento...,requirementsprior work as a project manager 4 ...,NaN,0,1,1
633,entry level,contract,customer service,consumer services,helpdesk associate,csl,welcome to the job portal designed for tedpb ...,high school or equivalent,100000130000,634,mu qb ebene,about our companycsl a wholly owned subsidiary...,you will need to haveexcellent information tec...,you will be provided withonthejob traininginsu...,0,1,1
5645,midsenior level,fulltime,marketing,unknown,experienced male caregivers needed todaythe be...,sales,missiongoldleaf homecare is revolutionizing ho...,NaN,100000130000,5646,us co denver,we take great care of our carepartners so they...,required qualification current cpr amp first a...,goldleaf provides competitive rates for caregi...,0,1,1
14369,internship,temporary,engineering,marketing and advertising,web developer internship,web analytics,we are a startup digital agency that is helpin...,bachelors degree,100000130000,14370,gr i kifisia athens,you have to do an internship as part of your u...,you must haveweb design coding skills eg htmlc...,on top of working for a fastgrowing agency tha...,0,1,1


In [ ]:
print(valores_mas_comunes(Xdes, 0, 20))
print(valores_mas_comunes(Xdes, 1, 20))
print(valores_mas_comunes(Xdes, 2, 20))
print(valores_mas_comunes(Xdes, 3, 20))
print(valores_mas_comunes(Xdes, 4, 20))
#print(valores_mas_comunes(Xdes, 5, 20))
#print(valores_mas_comunes(Xdes, 6, 20))
#print(valores_mas_comunes(Xdes, 7, 20))
print(valores_mas_comunes(Xdes, 8, 20))
print(valores_mas_comunes(Xdes, 9, 20))
print(valores_mas_comunes(Xdes, 10, 20))
print(valores_mas_comunes(Xdes, 11, 20))
print(contar_vacios(Xdes,8))